In [1]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.api as sm

In [4]:
# Scenario 4: Revenue Leakage & Fee Reconciliation.

# Calculate Discrepancy & Leakage: 
# Compare internal_clearing_fee against custodian_charged_fee to isolate overcharges vs. undercharges.   

# Quantify Financial Exposure: 
# Determine total net leakage, gross discrepancy, and leakage percentage across asset classes and custodians.

# Audit Flagging: 
# Tag trades with discrepancy severity (No Discrepancy, Minor Variance, High Leakage Risk) so operations 
# teams can prioritize recovery.

In [3]:

# 1. Connect to Database
db_path = (r"D:\IB Analytics CX\investment_bank_ops.db")
conn = sqlite3.connect(db_path)

# 2. Load fact_trades and dim_clients
trades_df = pd.read_sql_query("SELECT * FROM fact_trades", conn)
clients_df = pd.read_sql_query("SELECT client_id, client_name, client_tier FROM dim_clients", conn)

# 3. Calculate Fee Discrepancies & Revenue Leakage
# Discrepancy = Custodian Fee - Internal Expected Fee
trades_df["fee_discrepancy"] = (trades_df["custodian_charged_fee"] - trades_df["internal_clearing_fee"])
trades_df["abs_discrepancy"] = trades_df["fee_discrepancy"].abs()

# Flag Fee Mismatch (Threshold: > $5.00 variance)
trades_df["is_fee_mismatch"] = (trades_df["abs_discrepancy"] > 5.0).astype(int)

# Categorize Discrepancy Severity for Audit Desk
conditions = [
    (trades_df["fee_discrepancy"] > 5.0),  # Custodian Overcharged Us / Client
    (trades_df["fee_discrepancy"] < -5.0),  # We Undercharged (Uncollected Revenue)
    (trades_df["abs_discrepancy"] <= 5.0),  # Match / Acceptable Tolerance
]
choices = ["Custodian Overcharge", "Revenue Leakage (Undercharge)", "Matched"]
trades_df["discrepancy_category"] = np.select(
    conditions, choices, default="Matched"
)

# 4. Generate Summary Analytics for Verification
total_trades = len(trades_df)
mismatch_trades = trades_df["is_fee_mismatch"].sum()
total_leakage = trades_df[trades_df["fee_discrepancy"] < -5.0]["abs_discrepancy"].sum()
total_overcharge = trades_df[trades_df["fee_discrepancy"] > 5.0]["fee_discrepancy"].sum()

print("--- SCENARIO 4: REVENUE LEAKAGE & FEE RECONCILIATION SUMMARY ---")
print(f"Total Trades Analyzed: {total_trades:,}")
print(f"Trades with Fee Discrepancy (> $5): {mismatch_trades:,} ({(mismatch_trades/total_trades)*100:.2f}%)")
print(f"Total Uncollected Revenue Leakage: ${total_leakage:,.2f}")  # Money left on table
print(f"Total Custodian Overcharges: ${total_overcharge:,.2f}")  # Money to claim back

# 5. Write Updated Fact Table Back to Database
trades_df.to_sql("fact_trades", conn, if_exists="replace", index=False)

# Close Connection
conn.close()
print("\nSuccessfully updated 'fact_trades' table with Scenario 4 calculations!")

--- SCENARIO 4: REVENUE LEAKAGE & FEE RECONCILIATION SUMMARY ---
Total Trades Analyzed: 10,000
Trades with Fee Discrepancy (> $5): 698 (6.98%)
Total Uncollected Revenue Leakage: $0.00
Total Custodian Overcharges: $32,545.77

Successfully updated 'fact_trades' table with Scenario 4 calculations!


In [ ]:
# Scenario 4 Summary Analysis
# 6.98% Mismatch Rate (698 trades): A ~7% discrepancy rate across 10,000 trades is realistic for institutional 
# reconciliation desks (e.g., clearing variances between internal engines and external custodian statements).

# $32,545.77 Total Custodian Overcharges: $0.00 in uncollected revenue leakage means our internal billing engine isn't 
# undercharging clients; rather, custodian banks are systematically overcharging us on clearing fees.

# Operational Action Item: This $32.5k represents direct operational recovery value—a dispute list to be sent to 
# custodian banks for clawbacks.

In [ ]:
# Let's find where this $32.5k overcharge is concentrated: as per below

In [5]:

conn = sqlite3.connect(r"D:\IB Analytics CX\investment_bank_ops.db")

# Load updated fact_trades and dim_clients
query = """
SELECT 
    t.asset_class,
    t.discrepancy_category,
    COUNT(t.trade_id) AS mismatch_trade_count,
    SUM(t.fee_discrepancy) AS total_overcharge_amount,
    AVG(t.fee_discrepancy) AS avg_overcharge_per_trade
FROM fact_trades t
WHERE t.is_fee_mismatch = 1
GROUP BY t.asset_class, t.discrepancy_category
ORDER BY total_overcharge_amount DESC;
"""

breakdown_df = pd.read_sql_query(query, conn)
conn.close()

print("--- OVERCHARGE BREAKDOWN BY ASSET CLASS ---")
print(breakdown_df.to_string(index=False))

--- OVERCHARGE BREAKDOWN BY ASSET CLASS ---
    asset_class discrepancy_category  mismatch_trade_count  total_overcharge_amount  avg_overcharge_per_trade
       Equities Custodian Overcharge                   298                 14343.27                 48.131779
   Fixed Income Custodian Overcharge                   184                  7943.43                 43.170815
OTC Derivatives Custodian Overcharge                   152                  7166.92                 47.150789
     FX Options Custodian Overcharge                    64                  3092.15                 48.314844


In [ ]:
# Equities Drive Over 44% of Discrepancies: With $14,343.27 across 298 trades, Equities is the highest-volume leakage 
# point for custodian overcharging.

# Consistent Unit Impact: The average overcharge per trade sits tightly between $43.17 and $48.31 across all asset 
# classes, proving that custodian fee discrepancies aren't driven by isolated extreme outliers, but rather by a 
# systematic misapplication of fee schedules (e.g., wrong tiered bracket applied during clearing).

In [ ]:
# Scenario 5: Trade Volume & Settlement Forecasting

In [8]:
import numpy as np
import pandas as pd
import sqlite3
from sklearn.linear_model import LinearRegression

# 1. Connect to Database
db_path = (r"D:\IB Analytics CX\investment_bank_ops.db")
conn = sqlite3.connect(db_path)

# 2. Load fact_trades date & trade volume
query = """
SELECT 
    trade_date,
    COUNT(trade_id) AS actual_trade_volume
FROM fact_trades
GROUP BY trade_date
ORDER BY trade_date ASC;
"""
daily_vol_df = pd.read_sql_query(query, conn)

# Ensure datetime format
daily_vol_df["trade_date"] = pd.to_datetime(daily_vol_df["trade_date"])
daily_vol_df["day_index"] = np.arange(len(daily_vol_df))

# 3. Train OLS Volume Trend Model
X = daily_vol_df[["day_index"]]
y = daily_vol_df["actual_trade_volume"]

model = LinearRegression()
model.fit(X, y)

# 4. Generate 30-Day Forward Forecast (Using DataFrame to prevent feature name warnings)
last_day_index = daily_vol_df["day_index"].max()
last_date = daily_vol_df["trade_date"].max()

future_indices = pd.DataFrame({"day_index": np.arange(last_day_index + 1, last_day_index + 31)})

future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30)

forecast_vol = model.predict(future_indices)

# Combine Actuals and Forecast into a Unified Dataframe
forecast_df = pd.DataFrame(
    {
        "trade_date": future_dates,
        "actual_trade_volume": [np.nan] * 30,
        "forecast_trade_volume": np.round(forecast_vol, 0),
        "is_forecast": 1,
    }
)

daily_vol_df["forecast_trade_volume"] = np.round(model.predict(X), 0)
daily_vol_df["is_forecast"] = 0

unified_vol_df = pd.concat(
    [
        daily_vol_df[
            [
                "trade_date",
                "actual_trade_volume",
                "forecast_trade_volume",
                "is_forecast",
            ]
        ],
        forecast_df,
    ],
    ignore_index=True,
)

# Convert trade_date back to string for SQLite compatibility
unified_vol_df["trade_date"] = unified_vol_df["trade_date"].dt.strftime("%Y-%m-%d")

# 5. Save Forecast Results to Database
unified_vol_df.to_sql("fact_volume_forecast", conn, if_exists="replace", index=False)

print("--- SCENARIO 5: TRADE VOLUME FORECASTING SUMMARY ---")
print(f"Historical Days Analyzed: {len(daily_vol_df)}")
print(f"Average Daily Volume (Historical): {daily_vol_df['actual_trade_volume'].mean():.1f} trades/day")
print(f"30-Day Projected Avg Volume: {forecast_vol.mean():.1f} trades/day")
print(f"Model Slope (Growth Rate): {model.coef_[0]:.4f} trades/day")

conn.close()
print("\nSuccessfully created 'fact_volume_forecast' table in database!")

--- SCENARIO 5: TRADE VOLUME FORECASTING SUMMARY ---
Historical Days Analyzed: 365
Average Daily Volume (Historical): 27.4 trades/day
30-Day Projected Avg Volume: 27.2 trades/day
Model Slope (Growth Rate): -0.0010 trades/day

Successfully created 'fact_volume_forecast' table in database!


In [ ]:
# Key Findings:
# Historical Baseline: Over the last 365 days, trade volume averaged 27.4 trades/day.
# Projected Volume: Over the next 30 days, projected volume averages 27.2 trades/day.
# Model Slope (beta_1 = -0.0010): The linear trend shows a virtually flat trajectory 
# (a negligible decrease of ~0.03 trades per month).

# Operational Takeaway for Capital & Settlement Desks:
# Settlement volume is highly stable and predictable with zero structural spikes or systemic 
# growth pressure over the next 30 days.

# Capacity planning for clearing desks and custodian funding requirements can remain fixed at baseline 
# operational headcount without risk of volume-driven settlement bottlenecks.

In [ ]:
# Scenario 6: Settlement Penalty & Regulatory Capital Impact Simulation

In [11]:
# 1. Connect to Database
db_path = (r"D:\IB Analytics CX\investment_bank_ops.db")
conn = sqlite3.connect(db_path)

# 2. Load fact_trades
trades_df = pd.read_sql_query("SELECT * FROM fact_trades", conn)

# 3. Define CSDR / T+1 Basis Point Penalty Rates
# 1 bps = 0.0001 (0.01%)
penalty_rates = {
    "Equities": 0.0001,  # 1.0 bps
    "Fixed Income": 0.00005,  # 0.5 bps
    "OTC Derivatives": 0.00015,  # 1.5 bps
    "FX Options": 0.00015,  # 1.5 bps
}

# Map Penalty Rates to Each Trade
trades_df["penalty_rate_bps"] = (trades_df["asset_class"].map(penalty_rates).fillna(0.0001))

# 4. Calculate Simulated Penalty Exposure for Failed Trades
# Penalty = trade_amount * penalty_rate * settlement_delay_days (if trade_failed == 1)
trades_df["simulated_penalty_usd"] = np.where(
    trades_df["trade_failed"] == 1,
    trades_df["notional_amount"]
    * trades_df["penalty_rate_bps"]
    * trades_df["settlement_delay_days"],
    0.0,
)

# Round penalty to 2 decimal places
trades_df["simulated_penalty_usd"] = trades_df["simulated_penalty_usd"].round(2)

# 5. Generate Scenario 6 Summary Analytics
failed_trades_df = trades_df[trades_df["trade_failed"] == 1]

total_failed = len(failed_trades_df)

total_penalty_exposure = failed_trades_df["simulated_penalty_usd"].sum()

avg_penalty_per_failed_trade = failed_trades_df["simulated_penalty_usd"].mean()

print("--- SCENARIO 6: SETTLEMENT PENALTY & CSDR IMPACT SIMULATION SUMMARY ---")
print(f"Total Failed Trades Analyzed: {total_failed:,}")
print(f"Total Simulated CSDR/T+1 Penalty Cost: ${total_penalty_exposure:,.2f}")
print(f"Average Penalty per Failed Trade: ${avg_penalty_per_failed_trade:,.2f}")

# Asset Class Penalty Breakdown
penalty_by_asset = (
    failed_trades_df.groupby("asset_class")["simulated_penalty_usd"]
    .agg(["count", "sum", "mean"])
    .rename(
        columns={
            "count": "failed_trades",
            "sum": "total_penalty",
            "mean": "avg_penalty",
        }
    )
    .sort_values(by="total_penalty", ascending=False)
)

print("\n--- PENALTY EXPOSURE BY ASSET CLASS ---")
print(penalty_by_asset.to_string())

# 6. Save Updated Data Back to Database
trades_df.to_sql("fact_trades", conn, if_exists="replace", index=False)
conn.close()

print("\nSuccessfully updated 'fact_trades' with Scenario 6 penalty simulation!")

--- SCENARIO 6: SETTLEMENT PENALTY & CSDR IMPACT SIMULATION SUMMARY ---
Total Failed Trades Analyzed: 1,626
Total Simulated CSDR/T+1 Penalty Cost: $153,372.06
Average Penalty per Failed Trade: $94.32

--- PENALTY EXPOSURE BY ASSET CLASS ---
                 failed_trades  total_penalty  avg_penalty
asset_class                                               
OTC Derivatives            488       59815.55   122.572848
Equities                   559       56123.46   100.399750
Fixed Income               457       20741.06    45.385252
FX Options                 122       16691.99   136.819590

Successfully updated 'fact_trades' with Scenario 6 penalty simulation!


In [ ]:
# operational breakdown
# $153,372.06 Total Regulatory Capital Risk: Across 1,626 failed trades, regulatory penalty exposure totals ~$153.4k.

# OTC Derivatives Drive Highest Capital Drag: Despite having fewer fails than Equities (488 vs. 559), OTC Derivatives 
# carry the highest total penalty ($59,815.55) and a high unit penalty ($122.57/trade), driven by higher regulatory 
# bps rates and larger notional values.

# FX Options Highest Unit Risk: FX Options carry the highest per-trade failure cost ($136.82/trade), making late FX 
# settlements disproportionately expensive.